
# Lab 08: Getting Started with Matplotlib — Basic Plots & Common Patterns

**Duration:** ~45 minutes  
**Dataset:** `SampleData.csv`

## Learning Objectives
- Load a CSV and prep it for plotting
- Make essential Matplotlib charts (line, bar, histogram, scatter, box)
- Use common layout/formatting patterns (labels, titles, grids, legends, tick formatting)
- Save figures to disk

> Tip: The dataset looks like monthly utility billing/usage data with columns such as `Invoice_date`, `Billed_usage_kwh`, `Bill`, `Account_no`, `ProductID`, etc.


## Part 0 — Setup (5 min)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, MonthLocator

# Load CSV (path is set for this environment)
df = pd.read_csv("SampleData.csv",
    parse_dates=["Invoice_date"],
    dtype={"Account_no": 'string'})

# Parse and sort dates
df = df.sort_values("Invoice_date").reset_index(drop=True)

# Helpful time features
df["Year"] = df["Invoice_date"].dt.year
df["Month"] = df["Invoice_date"].dt.month
df["YearMonth"] = df["Invoice_date"].dt.to_period("M").dt.to_timestamp()
df["Quarter"] = df["Invoice_date"].dt.to_period("Q")

# Quick checks
display(df.head())
display(df.dtypes)
display(df[["Billed_usage_kwh","Bill"]].describe())


## Part A — Single-Series Line Plot (5–7 min)

In [ ]:

# Aggregate to monthly totals
monthly_kwh = (
    df.groupby("YearMonth", as_index=False)["Billed_usage_kwh"]
      .sum()
      .rename(columns={"Billed_usage_kwh":"Total_kwh"})
)
display(monthly_kwh.head())


In [ ]:

# Line chart with basic date formatting
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(monthly_kwh["YearMonth"], monthly_kwh["Total_kwh"], marker="o")
ax.set_title("Total Monthly Usage (kWh)")
ax.set_xlabel("Month")
ax.set_ylabel("kWh")

# Nice monthly ticks + readable dates
ax.xaxis.set_major_locator(MonthLocator(interval=2))
ax.xaxis.set_major_formatter(DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## Part B — Multiple Lines on One Axes (7–8 min)

In [ ]:

# Pivot to wide format (rows = months, cols = ProductID)
prod_month = (
    df.groupby(["YearMonth","ProductID"], as_index=False)["Billed_usage_kwh"]
      .sum()
      .pivot(index="YearMonth", columns="ProductID", values="Billed_usage_kwh")
      .fillna(0.0)
)
display(prod_month.head())


In [ ]:

# Plot each product as a line
fig, ax = plt.subplots(figsize=(9, 4.5))

for col in prod_month.columns:
    ax.plot(prod_month.index, prod_month[col], marker="o", label=f"Product {col}")

ax.set_title("Monthly Usage by Product")
ax.set_xlabel("Month")
ax.set_ylabel("kWh")
ax.xaxis.set_major_locator(MonthLocator(interval=2))
ax.xaxis.set_major_formatter(DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(title="Product", ncols=2, frameon=False)
plt.tight_layout()
plt.show()


## Part C — Bar Charts (Grouped) (7–8 min)

In [ ]:

# Aggregate by quarter × product
q_prod = (
    df.groupby(["Quarter","ProductID"], as_index=False)["Billed_usage_kwh"]
      .sum()
      .rename(columns={"Billed_usage_kwh":"kwh"})
)

# Make quarters chronological and pivot for grouped bars
q_prod_p = q_prod.pivot(index="Quarter", columns="ProductID", values="kwh").fillna(0.0)
q_labels = q_prod_p.index.astype(str).tolist()
display(q_prod_p.head())


In [ ]:

# Grouped bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))

n_groups = len(q_prod_p)
n_series = len(q_prod_p.columns)
x = np.arange(n_groups)
width = 0.8 / max(n_series, 1)

for i, col in enumerate(q_prod_p.columns):
    ax.bar(x + i*width, q_prod_p[col].values, width=width, label=f"Product {col}")

ax.set_title("Quarterly Usage by Product")
ax.set_xlabel("Quarter")
ax.set_ylabel("kWh")
ax.set_xticks(x + (n_series-1)*width/2)
ax.set_xticklabels(q_labels, rotation=0)

ax.grid(True, axis="y", linestyle="--", alpha=0.5)
ax.legend(title="Product", frameon=False)
plt.tight_layout()
plt.show()


## Part D — Histogram + Box Plot (6–7 min)

In [ ]:

# Histogram of Bill
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["Bill"], bins=20)
ax.set_title("Distribution of Bill Amounts")
ax.set_xlabel("Bill ($)")
ax.set_ylabel("Frequency")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:

# Box plot of Billed_usage_kwh by ProductID
usage_by_product = [df.loc[df["ProductID"]==pid, "Billed_usage_kwh"].values
                    for pid in sorted(df["ProductID"].unique())]

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(usage_by_product, showmeans=True)
ax.set_title("Usage (kWh) by Product")
ax.set_xlabel("Product")
ax.set_ylabel("kWh")
ax.set_xticklabels(sorted(df["ProductID"].unique()))
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## Part E — Scatter Plot (with simple trendline) (5–6 min)

In [ ]:

# Scatter: Billed_usage_kwh vs Bill
x = df["Billed_usage_kwh"].values
y = df["Bill"].values

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(x, y, alpha=0.7)
ax.set_title("Usage vs. Bill")
ax.set_xlabel("Usage (kWh)")
ax.set_ylabel("Bill ($)")
ax.grid(True, linestyle="--", alpha=0.5)

# Simple least-squares trendline (degree 1)
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = m * x_line + b
ax.plot(x_line, y_line)

plt.tight_layout()
plt.show()


## Part F — Subplots & Saving Figures (5 min)

In [ ]:

# Two-panel figure: histogram + scatter
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: Bill histogram
axes[0].hist(df["Bill"], bins=20)
axes[0].set_title("Bill Distribution")
axes[0].set_xlabel("Bill ($)")
axes[0].set_ylabel("Frequency")
axes[0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Right: Usage vs Bill scatter
axes[1].scatter(df["Billed_usage_kwh"], df["Bill"], alpha=0.7)
axes[1].set_title("Usage vs Bill")
axes[1].set_xlabel("Usage (kWh)")
axes[1].set_ylabel("Bill ($)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:

# Save the last figure to disk (re-run the cell above first if needed)
fig = plt.figure(figsize=(6,4))
plt.plot([0,1,2],[0,1,0])
plt.title("Example Save")
plt.tight_layout()
fig.savefig("matplotlib_summary.png", dpi=150, bbox_inches="tight")
print("Saved to matplotlib_summary.png")
plt.show()



## Quick Troubleshooting Notes
- **Date parsing errors:** Ensure `format="%m/%d/%Y"` matches the CSV. If unsure: `pd.to_datetime(df["Invoice_date"], errors="coerce")` and look for `NaT`.
- **Grouped bars misaligned:** Double-check bar centers (`x + i*width`) and matching `xticks`.
- **Tiny or crowded labels:** Use `plt.tight_layout()` and rotate ticks (`plt.xticks(rotation=45)`).
- **Trendline looks odd:** Remove outliers or try log-scaling (`ax.set_xscale('log')`) as an experiment.
